In [1]:
# python -m ipykernel install --user --name=askgen-venv --display-name "Python (AskGen)"

""" A executer danns le terminal pour creer un kernel pour jupyter notebook
ça permet de lancer le notebook avec les packages installés dans l'environnement virtuel askgen-venv
sans avoir à installer les packages dans l'environnement global de jupyter notebook """


" A executer danns le terminal pour creer un kernel pour jupyter notebook\nça permet de lancer le notebook avec les packages installés dans l'environnement virtuel askgen-venv\nsans avoir à installer les packages dans l'environnement global de jupyter notebook "

# Import des biblio

In [2]:


import sys
import os
import torch
import pandas as pd
from datasets import load_dataset

# Remonter d'un niveau pour accéder à la racine du projet AskGen
sys.path.append(os.path.abspath(os.path.join('..')))

from src.dataset import Vocab, QGDataset, collate_fn



print("Chemin de Python utilisé :", sys.executable)
print("Version de PyTorch :", torch.__version__)
print("Import Reussi !")

c:\Users\talha\Desktop\Cours-Dauphine\Semestre_3\NLP\AskGen\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chemin de Python utilisé : c:\Users\talha\Desktop\Cours-Dauphine\Semestre_3\NLP\AskGen\venv\Scripts\python.exe
Version de PyTorch : 2.14.1+cpu
Import Reussi !


# Visualisation
On prend juste 1000 lignes(observations) pour visualer

In [3]:
raw_train = load_dataset("rajpurkar/squad",split="train[:1000]",cache_dir="../data")
print(f"Nbre d'exples chargés: {len(raw_train)}")

Nbre d'exples chargés: 1000


In [4]:
df = raw_train.to_pandas()
print(df.shape)
df.head(5)


(1000, 5)


,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


In [5]:
# afficher juste les colonnes concernées
df[['context','question','answers']].head(3)

,context,question,answers
0,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."


In [6]:
# On initialise notre vocabulaire
vocab = Vocab()

# On rassemble tout le texte pour compter les mots
all_texts = [row['context'] + " " + row['question'] + " " + row['answers']['text'][0] for row in raw_train]

# On construit le vocabulaire (on garde les mots qui apparaissent au moins 2 fois)
vocab.build_vocab(all_texts)

print(f"Taille du vocabulaire : {len(vocab.word2id)} mots uniques.")
print("Exemple d'encodage :")
print("Mot 'Paris' ->", vocab.encode("Paris"))
print("ID 5 ->", vocab.decode([5]))

Taille du vocabulaire : 3787 mots uniques.
Exemple d'encodage :
Mot 'Paris' -> [843]
ID 5 -> the


In [13]:
# creation de notre dataset pour l'entrainement
dataset = QGDataset(raw_train, vocab)  

# creation du dataloader pour l'entrainement
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, collate_fn=collate_fn) # batch_size=2 pour l'exemple = 2 phrases par lot

src_batch, tgt_batch = next(iter(dataloader))

print("--DIMENSIONS DES TENSEURS--")
print("\nSource (batch x seq_len) :", src_batch.shape)
print("Target (batch x seq_len) :", tgt_batch.shape)

print(vocab.decode(src_batch[1].tolist())[:1500],"...\n") # on affiche les 300 premiers caractères de la source pour le premier exemple du batch

print("-- Cible(La question à générer)")
print(vocab.decode(tgt_batch[1].tolist()))

--DIMENSIONS DES TENSEURS--

Source (batch x seq_len) : torch.Size([2, 100])
Target (batch x seq_len) : torch.Size([2, 16])
architecturally , the school has a catholic character . atop the main building 's gold dome is a golden statue of the virgin mary . immediately in front of the main building and facing it , is a copper statue of christ with arms upraised with the legend " venite ad me omnes " . next to the main building is the basilica of the sacred heart . immediately behind the basilica is the grotto , a marian place of prayer and reflection . it is a replica of the grotto at lourdes , <sep> a copper statue of christ ...

-- Cible(La question à générer)
<start> what is in front of the notre dame main building ? <end> <pad> <pad> <pad>
